# ViShell — Sinh lệnh bash từ yêu cầu tiếng Việt (NL2Bash-vi)

**Pipeline:** NL2SH-ALFA (EN) → dịch sang tiếng Việt bằng vLLM → tạo biến thể nhiễu → lọc → SFT LoRA `Qwen2.5-Coder-1.5B-Instruct` → đánh giá tĩnh (không chạy lệnh) → demo.

**Cách chạy**
1. Trên Kaggle: Settings → Accelerator → chọn **GPU T4 x2** (notebook chỉ dùng 1 GPU T4, không cần cấu hình đa GPU).
2. Chạy lần lượt từ trên xuống. Để `SMOKE = True` trước (toàn pipeline trên 50 mẫu) để bắt lỗi sớm, rồi đổi `SMOKE = False`.
3. Nếu runtime bị ngắt: kết nối lại, **chạy lại từ đầu**. Mọi bước đều resume — phần đã xong tự bỏ qua (dịch theo chunk, SFT theo checkpoint, đánh giá theo từng khối).

**Thiết kế GPU:** các bước nặng (dịch bằng vLLM, SFT, sinh dự đoán) chạy trong **tiến trình con**. Khi tiến trình kết thúc, GPU được trả lại 100% — tránh lỗi quen thuộc vLLM không nhả VRAM trong cùng kernel.

### Ước lượng thời gian (Kaggle T4 16GB)

| Bước | Full | SMOKE |
|---|---|---|
| Phase 0 — cài đặt, kiểm tra thư mục Kaggle working | 5–8 phút | 5–8 phút |
| 1.1 Tải + làm sạch dữ liệu | 1–3 phút | < 1 phút |
| 1.2 Dịch EN→VI (~41k câu, Qwen3.5-4B) | 60–100 phút (gồm ~5 phút tải model) | 12–20 phút (chủ yếu tải model) |
| 1.3–1.4 Nhiễu + lọc | 1–3 phút | < 1 phút |
| 1.5 SFT LoRA 2 epoch (~52k mẫu, fp16) | 70–110 phút | 4–8 phút |
| 1.6 Đánh giá (base vs SFT × clean/noisy/EN) | 10–18 phút | 4–8 phút |
| 1.7 Demo | ~1 phút | ~1 phút |
| **Tổng** | **~2,5–4 giờ** | **~25–40 phút** |

T4 (Turing, không có nhân bf16) chậm hơn A100 khoảng 2–2,5 lần và bắt buộc dùng **fp16** thay bf16 — notebook đã cấu hình sẵn fp16 cho cả dịch (vLLM) và SFT. Kaggle giới hạn phiên GPU ~12 giờ/lần và ~30 giờ GPU/tuần; đây là ước lượng, thời gian thật từng bước ghi ở `logs/<run>/timings.jsonl`.

## PHASE 0 — Thiết lập
### 0.1 Cấu hình (chỉ cần sửa ô này)

In [1]:
# ================== CẤU HÌNH ==================
SMOKE = True        # True: chạy toàn pipeline trên 50 mẫu để kiểm tra. False: chạy thật.
N_SMOKE = 50
SEED = 42

ROOT = "/kaggle/working/vishell" # Changed for Kaggle compatibility
RUN = "smoke" if SMOKE else "full"   # smoke/full tách thư mục → smoke không làm bẩn dữ liệu resume của bản full

# Dataset đã kiểm tra trên Hugging Face: config "train" (~40.6k cặp, chưa kiểm định)
# và config "test" (300 câu, mỗi câu 2 lệnh đúng bash/bash2 = 600 cặp, kiểm tra thủ công — arXiv 2502.06858).
DATASET_ID = "westenfelder/NL2SH-ALFA"

# "auto": GPU >= 35GB (A100) → Qwen/Qwen3.5-9B ; GPU nhỏ hơn (L4, T4 16GB) → Qwen/Qwen3.5-4B.
# Dự phòng nếu bản vLLM không chạy được Qwen3.5: "Qwen/Qwen2.5-14B-Instruct" (A100) hoặc "Qwen/Qwen2.5-7B-Instruct" (L4/T4).
TRANSLATOR_MODEL = "auto"
SFT_BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

NOISE_RATIO = 0.30       # tỉ lệ mẫu train có bản nhiễu
NOISY_MODE = "augment"   # "augment": bản nhiễu là mẫu phụ (giữ cả bản sạch) | "replace": thay bản sạch
VAL_FRAC = 0.02          # tách val theo nhóm lệnh (chỉ để theo dõi eval loss)

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EPOCHS = 2
LR = 2e-4
MAX_LEN = 384

DEMO_INTERACTIVE = False  # True: bật vòng lặp input() ở ô demo

### 0.2 Kiểm tra GPU
Dùng `nvidia-smi` thay vì `torch` để kernel chính chưa khởi tạo CUDA (tiến trình con sẽ dùng trọn GPU).

In [2]:
import subprocess, json

def gpu_status():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True)
    if out.returncode != 0 or not out.stdout.strip():
        return None
    name, total, used = [x.strip() for x in out.stdout.strip().splitlines()[0].split(",")]
    return {"name": name, "total_gb": round(float(total) / 1024, 1), "used_gb": round(float(used) / 1024, 1)}

g = gpu_status()
assert g, "Không thấy GPU. Trên Kaggle: Settings → Accelerator → GPU T4 x2, rồi chạy lại."
print("GPU:", g)

VRAM_GB = g["total_gb"]
BIG_GPU = VRAM_GB >= 35
if TRANSLATOR_MODEL == "auto":
    TRANSLATOR_MODEL = "Qwen/Qwen3.5-9B" if BIG_GPU else "Qwen/Qwen3.5-4B"
TRANSLATOR_MULTIMODAL = any(k in TRANSLATOR_MODEL for k in ("Qwen3.5", "Qwen3.6"))  # Qwen3.5 có vision encoder → tắt khi chỉ dùng text

SMALL_VRAM = VRAM_GB < 20   # T4 16GB: batch nhỏ hơn để tránh OOM (L4 24GB đủ chỗ dùng batch lớn hơn)
TRAIN_BS = 32 if BIG_GPU else (8 if SMALL_VRAM else 16)
GRAD_ACC = 2 if BIG_GPU else (8 if SMALL_VRAM else 4)   # batch hiệu dụng = 64 trên mọi loại GPU
print(f"Model dịch: {TRANSLATOR_MODEL} | batch SFT {TRAIN_BS}×{GRAD_ACC} | RUN={RUN}")
if not BIG_GPU:
    print("⚠ GPU < 35GB (vd. T4 16GB): mọi bước vẫn chạy (dùng fp16, không dùng bf16) nhưng chậm hơn ~2–2,5 lần so với A100.")

GPU: {'name': 'Tesla T4', 'total_gb': 15.0, 'used_gb': 0.0}
Model dịch: Qwen/Qwen3.5-4B | batch SFT 8×8 | RUN=smoke
⚠ GPU < 35GB (vd. T4 16GB): mọi bước vẫn chạy (dùng fp16, không dùng bf16) nhưng chậm hơn ~2–2,5 lần so với A100.


### 0.3 Cài thư viện
Dùng **trl + peft** thay vì unsloth: unsloth và vLLM mỗi bên ghim một phiên bản torch riêng, cài chung một runtime rất dễ vỡ. Với model 1.5B + LoRA, trl+peft đã đủ nhanh trên cả A100/L4 và T4 (T4 dùng fp16 thay bf16, chậm hơn nhưng vẫn ổn với model nhỏ này).
Cài một lệnh `pip` duy nhất để resolver chọn bộ phiên bản tương thích với vLLM. Có file đánh dấu nên chạy lại không cài lại (trừ khi runtime mới).
Nếu sau khi cài gặp lỗi import torch lạ: Restart kernel rồi chạy lại từ đầu (không cài lại).

In [3]:
import os, sys, time, importlib.metadata as md

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
MARK = "/kaggle/working/.vishell_installed"
PKGS = ["vllm", "trl>=0.20", "peft", "datasets", "accelerate", "bashlex", "pandas", "torchao>0.16.0"]
if not os.path.exists(MARK):
    t0 = time.time()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS], check=True)
    open(MARK, "w").write("ok")
    print(f"Cài xong sau {time.time() - t0:.0f}s")
else:
    print("Đã cài trong runtime này — bỏ qua.")

# Kaggle có sẵn torchao bản cũ (0.10.0) ngay cả khi mọi thứ khác đã cài (MARK đã có) —
# kiểm tra và nâng cấp riêng, độc lập với MARK, vì trl/SFTTrainer cần torchao > 0.16.0.
try:
    from packaging import version
    if version.parse(md.version("torchao")) <= version.parse("0.16.0"):
        raise md.PackageNotFoundError
except md.PackageNotFoundError:
    print("Nâng cấp torchao...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>0.16.0"], check=True)
    print("torchao:", md.version("torchao"), "— nếu import trl vẫn báo lỗi cũ: Restart kernel rồi chạy lại từ đầu.")

for p in ["vllm", "torch", "transformers", "trl", "peft", "datasets", "bashlex", "torchao"]:
    try:
        print(f"{p:>12}: {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p:>12}: CHƯA CÀI")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/7

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.6 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.2.1 which is incompatible.
dopamine-rl 4.1.2 requires gy

Cài xong sau 280s
        vllm: 0.30.0
       torch: 2.13.0
transformers: 5.17.0
         trl: 1.14.0
        peft: 0.21.0
    datasets: 5.0.1
     bashlex: 0.18
     torchao: 0.18.0


### 0.4 Kiểm tra thư mục Kaggle working
Tạo (nếu chưa có) các thư mục con trong `/kaggle/working` dùng để lưu data/checkpoint/kết quả/log — không cần mount Drive trên Kaggle.

In [4]:
from pathlib import Path

def ensure_drive():
    # /kaggle/working là thư mục ghi mặc định trên Kaggle — không cần mount Drive.
    if Path("/kaggle/working").exists():
        print("Kaggle working directory đã sẵn sàng.")
        return
    else:
        raise RuntimeError("Kaggle working directory không tìm thấy. Kiểm tra môi trường Kaggle.")

ensure_drive()

ROOT_P = Path(ROOT)
DATA, CKPT, RES, LOGS = (ROOT_P / d / RUN for d in ("data", "checkpoints", "results", "logs"))
for d in (DATA, CKPT, RES, LOGS):
    d.mkdir(parents=True, exist_ok=True)

SCRIPTS = Path("/kaggle/working/vishell_scripts")   # script sinh lại mỗi phiên, để trên đĩa local
SCRIPTS.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache")   # cache model trên đĩa local, không ghi lên Drive
print("ROOT:", ROOT_P)
for d in (DATA, CKPT, RES, LOGS):
    print("  ", d)

Kaggle working directory đã sẵn sàng.
ROOT: /kaggle/working/vishell
   /kaggle/working/vishell/data/smoke
   /kaggle/working/vishell/checkpoints/smoke
   /kaggle/working/vishell/results/smoke
   /kaggle/working/vishell/logs/smoke


### 0.5 Hàm tiện ích (dùng chung cho notebook và các script con)
- `read_jsonl / write_jsonl / write_json`: ghi nguyên tử (file tạm → rename), không để lại file hỏng khi bị ngắt giữa chừng.
- `ChunkStore`: chia dữ liệu thành chunk cố định; chunk nào có file là đã xong → resume.
- `latest_checkpoint`: tìm `checkpoint-N` mới nhất có `trainer_state.json` (tức đã lưu trọn vẹn).
- `free_gpu`, `gpu_used_gb`, `run_py` (chạy script trong tiến trình con, stream log ra notebook + file), `timed`.
- Hàm bash: `parse_ok` (bashlex, có timeout), `normalize_cmd`, `main_utility`, `extract_command`.
- Hàm kiểm tra bản dịch: `protected_tokens`, `clean_translation`, `check_translation`.

In [5]:
%%writefile /kaggle/working/vishell_scripts/vs_utils.py
"""Tiện ích dùng chung cho notebook ViShell và các script con."""
import os, re, sys, json, gc, time, glob, signal, threading, subprocess, unicodedata
from pathlib import Path
from contextlib import contextmanager

SYSTEM_PROMPT = ("Bạn là trợ lý dòng lệnh Linux. Hãy chuyển yêu cầu của người dùng thành đúng MỘT lệnh bash "
                 "trên một dòng. Chỉ trả về lệnh, không giải thích, không dùng markdown.")

# ------------------------------------------------------------------ I/O
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def _atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def write_jsonl(path, rows):
    _atomic_write_text(path, "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))

def write_json(path, obj):
    _atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=2))

def read_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, encoding="utf-8") as f:
        return json.load(f)

class ChunkStore:
    """Lưu kết quả theo chunk cố định [i*cs, (i+1)*cs). Chunk có file = đã xong."""
    def __init__(self, dirpath, n_items, chunk_size):
        self.dir = Path(dirpath)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.n, self.cs = int(n_items), int(chunk_size)
        meta = {"n_items": self.n, "chunk_size": self.cs}
        meta_p = self.dir / "meta.json"
        old = read_json(meta_p)
        if old is None:
            write_json(meta_p, meta)
        elif old != meta:
            raise ValueError(f"{self.dir} đã có dữ liệu với cấu hình khác ({old} != {meta}). "
                             "Xoá thư mục này hoặc giữ nguyên cấu hình để resume.")

    @property
    def n_chunks(self):
        return (self.n + self.cs - 1) // self.cs

    def bounds(self, i):
        return i * self.cs, min((i + 1) * self.cs, self.n)

    def path(self, i):
        return self.dir / f"chunk_{i:05d}.jsonl"

    def pending(self):
        return [i for i in range(self.n_chunks) if not self.path(i).exists()]

    def write(self, i, rows):
        lo, hi = self.bounds(i)
        assert len(rows) == hi - lo, (len(rows), hi - lo)
        write_jsonl(self.path(i), rows)

    def read_all(self):
        miss = self.pending()
        if miss:
            raise RuntimeError(f"Còn {len(miss)} chunk chưa xong, ví dụ {miss[:5]}")
        out = []
        for i in range(self.n_chunks):
            out.extend(read_jsonl(self.path(i)))
        return out

    def progress(self):
        return f"{self.n_chunks - len(self.pending())}/{self.n_chunks} chunk"

def latest_checkpoint(out_dir):
    best, best_step = None, -1
    for p in glob.glob(os.path.join(str(out_dir), "checkpoint-*")):
        m = re.search(r"checkpoint-(\d+)$", p)
        if m and os.path.exists(os.path.join(p, "trainer_state.json")):
            s = int(m.group(1))
            if s > best_step:
                best, best_step = p, s
    return best

# ------------------------------------------------------------------ GPU / tiến trình
def gpu_used_gb():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True)
        return float(out.stdout.strip().splitlines()[0]) / 1024
    except Exception:
        return float("nan")

def free_gpu(verbose=True):
    """Gọi sau khi `del model` để trả VRAM của kernel hiện tại."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except ImportError:
        pass
    if verbose:
        print(f"GPU đang dùng: {gpu_used_gb():.1f} GB")

def _log_timing(log_dir, name, seconds, extra=None):
    rec = {"step": name, "seconds": round(seconds, 1), "time": time.strftime("%Y-%m-%d %H:%M:%S")}
    if extra:
        rec.update(extra)
    with open(Path(log_dir) / "timings.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

@contextmanager
def timed(name, log_dir):
    t0 = time.time()
    yield
    dt = time.time() - t0
    _log_timing(log_dir, name, dt)
    print(f"⏱ {name}: {dt / 60:.1f} phút")

def run_py(script, cfg, log_dir, name):
    """Chạy script Python trong tiến trình con; stream stdout ra notebook và logs/<name>.log."""
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = log_dir / f"{name}_cfg.json"
    write_json(cfg_path, cfg)
    env = dict(os.environ)
    env["PYTHONPATH"] = str(Path(script).parent) + os.pathsep + env.get("PYTHONPATH", "")
    env.setdefault("TQDM_MININTERVAL", "15")
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    t0 = time.time()
    with open(log_dir / f"{name}.log", "a", encoding="utf-8") as lf:
        lf.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
        proc = subprocess.Popen([sys.executable, "-u", str(script), str(cfg_path)],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1, env=env)
        for line in proc.stdout:
            print(line, end="")
            lf.write(line)
        rc = proc.wait()
    dt = time.time() - t0
    _log_timing(log_dir, name, dt, {"exit_code": rc})
    print(f"⏱ {name}: {dt / 60:.1f} phút | GPU sau bước: {gpu_used_gb():.1f} GB")
    if rc != 0:
        raise RuntimeError(f"Bước '{name}' lỗi (exit {rc}). Log: {log_dir / (name + '.log')}")
    return dt

def hf_dtype_kw(torch_dtype):
    """transformers >= 4.56 dùng `dtype`, bản cũ dùng `torch_dtype`."""
    import transformers
    from packaging import version
    key = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
    return {key: torch_dtype}

# ------------------------------------------------------------------ Bash
class _ParseTimeout(Exception):
    pass

def _on_alarm(signum, frame):
    raise _ParseTimeout()

def bash_parse(cmd, timeout=2.0):
    import bashlex
    use_alarm = hasattr(signal, "setitimer") and threading.current_thread() is threading.main_thread()
    old = None
    if use_alarm:
        old = signal.signal(signal.SIGALRM, _on_alarm)
        signal.setitimer(signal.ITIMER_REAL, timeout)
    try:
        return bashlex.parse(cmd)
    finally:
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old)

def parse_ok(cmd):
    if not cmd or not cmd.strip():
        return False
    try:
        bash_parse(cmd)
        return True
    except Exception:
        return False

def normalize_cmd(c):
    c = (c or "").strip()
    c = re.sub(r"\s+", " ", c)
    c = re.sub(r"\s*;\s*$", "", c)
    return c

WRAPPERS = {"sudo", "doas", "nohup", "time", "nice", "ionice", "env", "command", "builtin",
            "exec", "stdbuf", "timeout", "watch"}

def _first_command_words(node):
    if getattr(node, "kind", None) == "command":
        return [p.word for p in node.parts if getattr(p, "kind", None) == "word"]
    for attr in ("parts", "list"):
        for ch in (getattr(node, attr, None) or []):
            if hasattr(ch, "kind"):
                r = _first_command_words(ch)
                if r:
                    return r
    return None

def _pick_utility(words):
    i = 0
    while i < len(words):
        base = os.path.basename(words[i])
        if base in WRAPPERS:
            i += 1
            while i < len(words) and (words[i].startswith("-") or re.match(r"^[A-Za-z_]\w*=", words[i])
                                      or (base == "timeout" and re.match(r"^\d", words[i]))):
                i += 1
            continue
        if re.match(r"^[A-Za-z_]\w*=", words[i]):
            i += 1
            continue
        return base or None
    return None

def main_utility(cmd):
    """Tiện ích chính = từ lệnh đầu tiên của simple command đầu tiên (bỏ sudo/env/time/..., bỏ gán biến)."""
    if not cmd or not cmd.strip():
        return None
    try:
        for tree in bash_parse(cmd):
            words = _first_command_words(tree)
            if words:
                u = _pick_utility(words)
                if u:
                    return u
    except Exception:
        pass
    first = re.split(r"\|\||&&|[|;&]", cmd, maxsplit=1)[0]
    return _pick_utility(first.split())

FENCE_RE = re.compile(r"```[a-zA-Z]*\s*\n?(.*?)```", re.S)

def extract_command(text):
    t = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()
    m = FENCE_RE.search(t)
    if m:
        t = m.group(1)
    for line in t.splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        s = re.sub(r"^\$\s+", "", s)
        if len(s) > 1 and s.startswith("`") and s.endswith("`"):
            s = s.strip("`").strip()
        return s
    return ""

# ------------------------------------------------------------------ Kiểm tra bản dịch
VI_CHARS = set("àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ")
CJK_RE = re.compile(r"[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7af]")
QUOTE_RE = re.compile(r"""(?:^|(?<=[\s(\[]))(['"`])(.+?)\1(?=$|[\s.,;:!?)])""")
ORD_RE = re.compile(r"^(\d+)(st|nd|rd|th)$", re.I)
ABBR_RE = re.compile(r"^(?:[a-z]\.)+[a-z]\.?$|^etc\.?$", re.I)
COMMON_EN = set("""a an the and or of to in on at by for with from into all any each every this that these those
file files directory directories folder folders line lines word words name names current path size time date
find list print show display remove delete copy move make create change set get count sort search replace
read write open close kill stop start run test type which who watch touch head tail cut join split paste
less more free top find yes no true false echo cat sleep wait read mail more help info man user users group
number numbers first last new old all only not recursively recursive""".split())

def protected_tokens(nl, bash=""):
    """hard: bắt buộc giữ nguyên (đường dẫn, tên file, số, cờ, biến, chuỗi trong nháy).
       soft: từ ASCII xuất hiện trong lệnh (thường là tên tiện ích) — chỉ ghi nhận, không loại."""
    hard, soft = [], []
    for _, inner in QUOTE_RE.findall(nl):
        if inner.strip():
            hard.append(inner.strip())
    bash_words = set(re.findall(r"[A-Za-z][A-Za-z0-9_+-]*", bash or ""))
    for tok in nl.split():
        t = tok.strip("\"'`()[]{},;:!?")
        if t not in (".", ".."):
            t = t.rstrip(".")
        if not t or re.fullmatch(r"[\W_]+", t):
            continue
        m = ORD_RE.match(t)
        if m:
            hard.append(m.group(1))
            continue
        if ABBR_RE.match(t):
            continue
        if re.search(r"[/._$~*=\d]", t) or re.match(r"^--?[A-Za-z0-9]", t):
            hard.append(t)
        elif t.isascii() and t.isalpha() and len(t) >= 2 and t in bash_words and t.lower() not in COMMON_EN:
            soft.append(t)
    dedup = lambda xs: list(dict.fromkeys(xs))
    return dedup(hard), dedup(soft)

PREFIX_RE = re.compile(r"^(?:bản dịch|dịch|tiếng việt|vietnamese|translation|câu tiếng việt)\s*[:：\-]\s*", re.I)

def clean_translation(text):
    t = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).replace("<think>", "").replace("</think>", "")
    lines = [l.strip() for l in t.strip().splitlines() if l.strip()]
    if not lines:
        return ""
    t = PREFIX_RE.sub("", lines[0]).strip()
    if len(t) >= 2 and t[0] in "\"“" and t[-1] in "\"”" and '"' not in t[1:-1]:
        t = t[1:-1].strip()
    return unicodedata.normalize("NFC", t)

def check_translation(en, vi, bash):
    """Trả về (ok, reason, soft_missing)."""
    if not vi:
        return False, "empty", []
    if CJK_RE.search(vi):
        return False, "cjk", []
    if not any(c in VI_CHARS for c in vi.lower()):
        en_w = set(re.findall(r"[a-z]+", en.lower()))
        vi_w = set(re.findall(r"[a-z]+", vi.lower()))
        if vi.strip().lower() == en.strip().lower() or (vi_w and len(vi_w & en_w) / len(vi_w) > 0.6):
            return False, "not_vietnamese", []
    ratio = len(vi) / max(1, len(en))
    if ratio < 0.4 or ratio > 4:
        return False, "length_ratio", []
    nb = normalize_cmd(bash)
    if len(nb) >= 8 and nb in vi and nb not in en:
        return False, "leaked_command", []
    hard, soft = protected_tokens(en, bash)
    miss = [t for t in hard if t not in vi]
    if miss:
        return False, "missing_token", []
    soft_miss = [t for t in soft if not re.search(r"(?<!\w)" + re.escape(t) + r"(?!\w)", vi)]
    return True, "ok", soft_miss

Writing /kaggle/working/vishell_scripts/vs_utils.py


In [6]:
import importlib, sys
sys.path.insert(0, str(SCRIPTS))
import vs_utils
importlib.reload(vs_utils)
from vs_utils import *

# kiểm tra nhanh các hàm bash
for c in ["sudo find . -name '*.log' | xargs rm", "FOO=1 env LANG=C sort a.txt", "ls -la &&", "echo $((1+2))"]:
    print(f"{c!r:45} parse={parse_ok(c)!s:5} utility={main_utility(c)}")
print(protected_tokens("print the last 5 lines of setup_nl2b_fs_1.sh", "tail -n 5 setup_nl2b_fs_1.sh"))
print(check_translation("print the 10th line of a.txt", "in ra dòng thứ 10 của a.txt", "sed -n 10p a.txt"))

"sudo find . -name '*.log' | xargs rm"        parse=True  utility=find
'FOO=1 env LANG=C sort a.txt'                 parse=True  utility=sort
'ls -la &&'                                   parse=False utility=ls
'echo $((1+2))'                               parse=False utility=echo
(['5', 'setup_nl2b_fs_1.sh'], [])
(True, 'ok', [])


## PHASE 1 — Dữ liệu và SFT
### 1.1 Tải dataset, làm sạch, chống rò rỉ train↔test  (~1–3 phút)
- **Test:** bộ test đã sửa lỗi của arXiv 2502.06858 (config `test` của `westenfelder/NL2SH-ALFA`): 300 câu, mỗi câu có 2 lệnh đúng (`bash`, `bash2`) và độ khó 0/1/2. Test **không** lọc bằng bashlex (lệnh đã được kiểm tra tay; bashlex không hỗ trợ hết cú pháp bash), chỉ báo cáo tỉ lệ parse được làm mốc.
- **Train:** bỏ trùng; bỏ mọi mẫu có lệnh trùng `bash`/`bash2` của test hoặc câu mô tả trùng test (vì train được gộp từ các bộ cũ mà test cũng lấy nguồn từ đó); bỏ lệnh nhiều dòng; bỏ lệnh không parse được bằng bashlex **trước khi dịch** để khỏi tốn GPU dịch mẫu sẽ bị loại.
- Nếu không tải được NL2SH-ALFA, ô sẽ rơi về `jiacheng-ye/nl2bash` và tách test **theo nhóm lệnh** (không lệnh nào của test xuất hiện trong train).

In [7]:
import random, collections
from tqdm.auto import tqdm
from datasets import load_dataset

def _pick_col(cols, cands):
    for c in cands:
        if c in cols:
            return c
    raise KeyError(f"Không tìm thấy cột trong {cols}")

with timed("1.1_load_data", LOGS):
    try:
        ds_train = load_dataset(DATASET_ID, "train", split="train")   # lưu ý: config chọn train/test, split luôn là "train"
        ds_test = load_dataset(DATASET_ID, "test", split="train")
        TEST_SOURCE = f"{DATASET_ID} [test] — kiểm tra thủ công (arXiv 2502.06858)"
        train_raw = [{"nl": r["nl"], "bash": r["bash"]} for r in ds_train]
        test_raw = [{"nl": r["nl"], "bash": r["bash"], "bash2": r.get("bash2") or r["bash"],
                     "difficulty": int(r.get("difficulty", -1))} for r in ds_test]
    except Exception as e:
        print("⚠ Không tải được NL2SH-ALFA:", repr(e), "→ dùng jiacheng-ye/nl2bash và tách test theo nhóm lệnh")
        dsd = load_dataset("jiacheng-ye/nl2bash")
        allrows = []
        for split in dsd:
            cols = dsd[split].column_names
            cn, cc = _pick_col(cols, ["nl", "invocation", "description", "text", "input"]), \
                     _pick_col(cols, ["bash", "cmd", "command", "code", "output"])
            allrows += [{"nl": r[cn], "bash": r[cc]} for r in dsd[split]]
        groups = sorted({normalize_cmd(r["bash"]) for r in allrows})
        random.Random(SEED).shuffle(groups)
        test_groups = set(groups[: max(1, int(0.05 * len(groups)))])
        test_raw = [{**r, "bash2": r["bash"], "difficulty": -1} for r in allrows if normalize_cmd(r["bash"]) in test_groups]
        train_raw = [r for r in allrows if normalize_cmd(r["bash"]) not in test_groups]
        TEST_SOURCE = "jiacheng-ye/nl2bash — tách theo nhóm lệnh (fallback)"

    stats = {"test_source": TEST_SOURCE, "train_raw": len(train_raw), "test_raw": len(test_raw)}

    # --- làm sạch train
    test_cmds = {normalize_cmd(r["bash"]) for r in test_raw} | {normalize_cmd(r["bash2"]) for r in test_raw}
    test_nls = {r["nl"].strip().lower() for r in test_raw}
    seen, train_clean, drop = set(), [], collections.Counter()
    for r in train_raw:
        nl, cmd = (r["nl"] or "").strip(), (r["bash"] or "").strip()
        if not nl or not cmd:
            drop["empty"] += 1; continue
        if "\n" in cmd:
            drop["multiline"] += 1; continue
        if len(cmd) > 300 or len(nl) > 400:
            drop["too_long"] += 1; continue
        nc = normalize_cmd(cmd)
        if nc in test_cmds or nl.lower() in test_nls:
            drop["overlap_test"] += 1; continue
        key = (nl.lower(), nc)
        if key in seen:
            drop["duplicate"] += 1; continue
        seen.add(key)
        train_clean.append({"nl": nl, "bash": cmd})

    ok_rows = []
    for r in tqdm(train_clean, desc="bashlex train", mininterval=5):
        if parse_ok(r["bash"]):
            ok_rows.append(r)
        else:
            drop["bashlex_fail"] += 1
    train_clean = ok_rows
    gold_parse = sum(parse_ok(r["bash"]) for r in test_raw) / max(1, len(test_raw))

    rng = random.Random(SEED)
    rng.shuffle(train_clean)
    if SMOKE:
        train_clean = train_clean[:N_SMOKE]
        test_raw = random.Random(SEED).sample(test_raw, min(N_SMOKE, len(test_raw)))
    en_test = [{"id": f"te-{i:04d}", **r} for i, r in enumerate(test_raw)]
    en_train = [{"id": f"tr-{i:06d}", **r} for i, r in enumerate(train_clean)]

    stats.update({"train_after_clean": len(en_train), "test_used": len(en_test),
                  "train_dropped": dict(drop), "test_gold_bashlex_parse_rate": round(gold_parse, 4)})
    write_jsonl(DATA / "en_train.jsonl", en_train)
    write_jsonl(DATA / "en_test.jsonl", en_test)
    write_json(RES / "data_stats.json", stats)

print(json.dumps(stats, ensure_ascii=False, indent=2))
print("Ví dụ test:", en_test[0])

README.md:   0%|          | 0.00/4.76k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/4.91M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40639 [00:00<?, ? examples/s]

test.csv:   0%|          | 0.00/49.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

bashlex train:   0%|          | 0/40447 [00:00<?, ?it/s]

⏱ 1.1_load_data: 0.3 phút
{
  "test_source": "westenfelder/NL2SH-ALFA [test] — kiểm tra thủ công (arXiv 2502.06858)",
  "train_raw": 40639,
  "test_raw": 300,
  "train_after_clean": 50,
  "test_used": 50,
  "train_dropped": {
    "overlap_test": 73,
    "duplicate": 106,
    "too_long": 13,
    "bashlex_fail": 115
  },
  "test_gold_bashlex_parse_rate": 0.9967
}
Ví dụ test: {'id': 'te-0000', 'nl': 'print the bash profile in the home directory', 'bash': 'cat ~/.bashrc', 'bash2': 'nl ~/.bashrc', 'difficulty': 0}


### 1.2 Dịch EN → VI bằng vLLM  (A100: ~25–45 phút; T4 16GB: ~60–100 phút cho ~41k câu; SMOKE: ~6–20 phút, chủ yếu tải model)
- Model: **Qwen3.5-9B** trên GPU ≥35GB (A100), **Qwen3.5-4B** trên GPU nhỏ hơn (L4, T4 16GB) — chọn tự động theo VRAM. vLLM chạy **fp16** (T4 không có nhân bf16). Qwen3.5 mặc định "suy nghĩ" trước khi trả lời → tắt bằng `enable_thinking=False`; tắt luôn vision encoder vì chỉ dùng text.
- Prompt có vài ví dụ mẫu và kèm lệnh bash tham chiếu (chỉ để hiểu ngữ cảnh). Mỗi bản dịch được kiểm tra: không rỗng, không lẫn chữ Trung/Nhật/Hàn, đúng là tiếng Việt, độ dài hợp lý, không chép nguyên lệnh, **giữ nguyên** đường dẫn / tên file / số / cờ / biến / chuỗi trong nháy. Không đạt → dịch lại (tối đa 3 lần, nhiệt độ tăng dần).
- Lưu theo chunk (2000 câu, SMOKE: 25 câu) vào `/kaggle/working`; bị ngắt thì chạy lại ô, chỉ dịch chunk còn thiếu. Test được đặt lên đầu để xong trước.

In [8]:
%%writefile /kaggle/working/vishell_scripts/translate.py
import sys, json
from vs_utils import read_jsonl, ChunkStore, clean_translation, check_translation

SYS = """Bạn là biên dịch viên kỹ thuật. Hãy dịch yêu cầu tiếng Anh sang tiếng Việt tự nhiên, giống cách một lập trình viên Việt Nam gõ yêu cầu cho trợ lý dòng lệnh.\nQuy tắc:\n1. GIỮ NGUYÊN, không dịch, không thêm dấu: tên file, đường dẫn, tên thư mục, tên chương trình/tiện ích (grep, find, tar...), tùy chọn (-l, --all), biến ($HOME), đuôi file (.txt), số, URL, tên người dùng/máy/gói, mẫu tìm kiếm.\n2. Giữ nguyên dấu nháy và toàn bộ nội dung bên trong dấu nháy.\n3. Không thêm, không bớt ý. Không giải thích. Không viết lệnh bash.\n4. Dùng thuật ngữ quen thuộc của dân IT Việt: file, thư mục, quyền, tiến trình, đường dẫn...\n5. Chỉ trả về MỘT dòng là bản dịch."""

FEWSHOT = [
    ("find all .txt files in /home/user larger than 10MB", 'find /home/user -name "*.txt" -size +10M',
     "tìm tất cả file .txt trong /home/user có kích thước lớn hơn 10MB"),
    ("print the last 5 lines of config_backup.sh", "tail -n 5 config_backup.sh",
     "in ra 5 dòng cuối của file config_backup.sh"),
    ("change the owner of /srv/app/data.db to user 'www-data'", "chown www-data /srv/app/data.db",
     "đổi chủ sở hữu của /srv/app/data.db thành người dùng 'www-data'"),
    ("Counts lines in all *.py files in the current directory tree", "find . -name '*.py' | xargs wc -l",
     "đếm số dòng của tất cả file *.py trong cây thư mục hiện tại"),
    ("Recursively removes all empty directories under current directory", "find . -type d -empty -delete",
     "xoá đệ quy tất cả thư mục rỗng trong thư mục hiện tại"),
]

def user_msg(nl, bash):
    return f"Câu tiếng Anh: {nl}\nLệnh tham chiếu (chỉ để hiểu ngữ cảnh, KHÔNG đưa vào bản dịch): {bash}"

def build_messages(r):
    msgs = [{"role": "system", "content": SYS}]
    for en, cmd, vi in FEWSHOT:
        msgs += [{"role": "user", "content": user_msg(en, cmd)}, {"role": "assistant", "content": vi}]
    msgs.append({"role": "user", "content": user_msg(r["nl"], r["bash"])})
    return msgs

def make_llm(cfg):
    from vllm import LLM
    kw = dict(model=cfg["model"], dtype="float16", max_model_len=cfg["max_model_len"],
              gpu_memory_utilization=cfg["gpu_mem"], seed=cfg["seed"],
              max_num_seqs=cfg.get("max_num_seqs", 128))   # giới hạn thấp cho VRAM nhỏ (T4): tránh lỗi thiếu Mamba cache block
    if cfg.get("multimodal"):
        try:
            return LLM(**kw, language_model_only=True)       # bỏ vision encoder, dành VRAM cho KV cache
        except TypeError:
            return LLM(**kw, limit_mm_per_prompt={"image": 0, "video": 0})
    return LLM(**kw)

def main():
    cfg = json.load(open(sys.argv[1]))
    rows = read_jsonl(cfg["input"])
    store = ChunkStore(cfg["out_dir"], len(rows), cfg["chunk_size"])
    pending = store.pending()
    print(f"[translate] {len(rows)} câu, {store.progress()} đã xong, còn {len(pending)} chunk", flush=True)
    if not pending:
        return
    from vllm import SamplingParams
    llm = make_llm(cfg)
    temps = cfg["temperatures"]
    for ci in pending:
        lo, hi = store.bounds(ci)
        batch = rows[lo:hi]
        res, todo = {}, batch
        for attempt, temp in enumerate(temps):
            if not todo:
                break
            sp = SamplingParams(temperature=temp, top_p=0.8, top_k=20, max_tokens=cfg["max_tokens"],
                                seed=cfg["seed"] + attempt)
            outs = llm.chat([build_messages(r) for r in todo], sp, use_tqdm=True,
                            chat_template_kwargs={"enable_thinking": False})
            nxt = []
            for r, o in zip(todo, outs):
                vi = clean_translation(o.outputs[0].text)
                ok, reason, soft_miss = check_translation(r["nl"], vi, r["bash"])
                res[r["id"]] = {"id": r["id"], "nl_vi": vi, "ok": ok, "reason": reason,
                                "soft_missing": soft_miss, "tries": attempt + 1, "model": cfg["model"]}
                if not ok:
                    nxt.append(r)
            todo = nxt
        store.write(ci, [res[r["id"]] for r in batch])
        n_ok = sum(res[r["id"]]["ok"] for r in batch)
        print(f"[translate] chunk {ci}: {n_ok}/{len(batch)} đạt | {store.progress()}", flush=True)

if __name__ == "__main__":
    main()

Writing /kaggle/working/vishell_scripts/translate.py


In [9]:
en_train = read_jsonl(DATA / "en_train.jsonl")
en_test = read_jsonl(DATA / "en_test.jsonl")
to_tr = en_test + en_train                     # test trước
TR_IN = DATA / "to_translate.jsonl"
if TR_IN.exists():
    old_ids = [r["id"] for r in read_jsonl(TR_IN)]
    assert old_ids == [r["id"] for r in to_tr], "to_translate.jsonl khác dữ liệu hiện tại — xoá data/<run>/translate* để dịch lại."
else:
    write_jsonl(TR_IN, to_tr)

TR_DIR = DATA / "translate_chunks"
tr_cfg = dict(model=TRANSLATOR_MODEL, multimodal=TRANSLATOR_MULTIMODAL, input=str(TR_IN), out_dir=str(TR_DIR),
              chunk_size=25 if SMOKE else 2000, max_model_len=2048, max_tokens=256,
              gpu_mem=0.90, seed=SEED, temperatures=[0.2, 0.7, 0.9],
              max_num_seqs=64 if SMALL_VRAM else 128)   # T4: giới hạn số seq đồng thời để tránh hết Mamba cache block (Qwen3.5)
run_py(SCRIPTS / "translate.py", tr_cfg, LOGS, "translate")

vi_by_id = {r["id"]: r for r in ChunkStore(TR_DIR, len(to_tr), tr_cfg["chunk_size"]).read_all()}
reasons = collections.Counter(r["reason"] for r in vi_by_id.values())
print("Kết quả kiểm tra bản dịch:", dict(reasons))
print("Số lần thử:", dict(collections.Counter(r["tries"] for r in vi_by_id.values())))
for r in random.Random(0).sample(to_tr, min(8, len(to_tr))):
    v = vi_by_id[r["id"]]
    print(f"\nEN : {r['nl']}\nVI : {v['nl_vi']}  [{v['reason']}]\nCMD: {r['bash']}")

[translate] 100 câu, 0/4 chunk đã xong, còn 4 chunk
W0926 07:05:36.085000 165 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0926 07:05:36.127000 165 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
INFO 09-26 07:05:44 [api_utils.py:286] non-default args: {'dtype': 'float16', 'seed': 42, 'max_model_len': 2048, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 64, 'disable_log_stats': True, 'language_model_only': True, 'model': 'Qwen/Qwen3.5-4B'}
INFO 09-26 07:06:04 [model.py:692] Resolved architecture: Qwen3_5ForConditionalGeneration
WARNING 09-26 07:06:04 [model.py:2364] Casting torch.bfl

#### 1.2b File rà soát bản dịch test (quan trọng cho paper)
Test dịch máy nên được người rà lại trước khi báo cáo số liệu. Ô dưới xuất `data/<run>/test_vi_review.csv`. Sửa trong Google Sheets, thêm cột `nl_vi_fixed` cho câu cần sửa, lưu thành **`test_vi_reviewed.csv`** cùng thư mục → lần chạy sau sẽ tự dùng bản đã sửa (các bước sau ô này tự tính lại).

In [10]:
import pandas as pd

rev_rows = [{"id": r["id"], "nl_en": r["nl"], "nl_vi": vi_by_id[r["id"]]["nl_vi"], "check": vi_by_id[r["id"]]["reason"],
             "bash": r["bash"], "bash2": r["bash2"], "difficulty": r["difficulty"]} for r in en_test]
pd.DataFrame(rev_rows).to_csv(DATA / "test_vi_review.csv", index=False, encoding="utf-8-sig")
print("Đã xuất", DATA / "test_vi_review.csv")

REVIEWED = DATA / "test_vi_reviewed.csv"
n_fixed = 0
if REVIEWED.exists():
    fixed = pd.read_csv(REVIEWED, dtype=str).fillna("")
    for _, row in fixed.iterrows():
        if row.get("nl_vi_fixed", "").strip() and row["id"] in vi_by_id:
            vi_by_id[row["id"]].update(nl_vi=row["nl_vi_fixed"].strip(), ok=True, reason="human_fixed")
            n_fixed += 1
print(f"Áp dụng {n_fixed} câu test đã sửa tay.")

Đã xuất /kaggle/working/vishell/data/smoke/test_vi_review.csv
Áp dụng 0 câu test đã sửa tay.


### 1.3 Tạo biến thể nhiễu  (< 1 phút)
Ba loại nhiễu, mỗi câu nhiễu lấy ngẫu nhiên 1–3 loại:
- **no_accent**: bỏ dấu toàn câu (70%) hoặc một phần số từ (30%) — kiểu gõ không dấu.
- **mix_en**: thay một số thuật ngữ tiếng Việt bằng tiếng Anh (`thư mục`→`folder`, `xoá`→`delete`...) — kiểu nói trộn của dân IT.
- **typo**: 1–2 lỗi gõ nhẹ (đảo, thiếu, lặp ký tự, gõ nhầm phím kề). Không đụng vào token có trong lệnh (tên file, tiện ích...).

Train: ~30% mẫu có bản nhiễu (chọn tất định theo id → chạy lại cho kết quả y hệt). Test: mọi câu đều có cả bản sạch và bản nhiễu.

In [11]:
import unicodedata, re

def strip_accents(s):
    s = s.replace("đ", "d").replace("Đ", "D")
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return unicodedata.normalize("NFC", s)

TERM_MIX = {
    "thư mục": ["folder", "directory", "dir"], "tệp tin": ["file"], "tập tin": ["file"], "tệp": ["file"],
    "xoá": ["delete", "remove"], "xóa": ["delete", "remove"], "tìm kiếm": ["search"], "tìm": ["find", "search"],
    "liệt kê": ["list"], "sao chép": ["copy"], "di chuyển": ["move"], "đổi tên": ["rename"],
    "quyền": ["permission"], "tiến trình": ["process"], "giải nén": ["unzip", "extract"], "nén": ["zip", "compress"],
    "dòng": ["line"], "đường dẫn": ["path"], "người dùng": ["user"], "mạng": ["network"], "ổ đĩa": ["disk"],
    "dung lượng": ["size"], "kích thước": ["size"], "hiển thị": ["show", "display"], "in ra": ["print"],
    "tải xuống": ["download"], "tải về": ["download"], "biến môi trường": ["env", "environment variable"],
    "chuỗi": ["string"], "cổng": ["port"], "đệ quy": ["recursive"], "rỗng": ["empty"], "thay thế": ["replace"],
    "sắp xếp": ["sort"], "đếm": ["count"], "lệnh": ["command"], "máy chủ": ["server"], "nhật ký": ["log"],
    "cấu hình": ["config"], "gói": ["package"], "tạo": ["create"], "chạy": ["run"], "dừng": ["stop", "kill"],
}
_TERM_PATS = [(re.compile(r"(?<!\w)" + re.escape(k) + r"(?!\w)", re.I), v)
              for k, v in sorted(TERM_MIX.items(), key=lambda kv: -len(kv[0]))]

_ROWS = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
NEIGHBORS = {}
for ri, row in enumerate(_ROWS):
    for ci, ch in enumerate(row):
        nb = [row[j] for j in (ci - 1, ci + 1) if 0 <= j < len(row)]
        for rj in (ri - 1, ri + 1):
            if 0 <= rj < len(_ROWS) and ci < len(_ROWS[rj]):
                nb.append(_ROWS[rj][ci])
        NEIGHBORS[ch] = nb

def mix_terms(s, rng, p=0.5):
    for pat, opts in _TERM_PATS:
        s = pat.sub(lambda m: rng.choice(opts) if rng.random() < p else m.group(0), s)
    return s

def partial_no_accent(s, rng):
    return " ".join(strip_accents(w) if rng.random() < 0.5 else w for w in s.split(" "))

def add_typos(s, rng, bash, n=1):
    bash_tokens = set(re.findall(r"\S+", bash))
    spans = [m.span() for m in re.finditer(r"\w+", s)
             if m.group(0).isalpha() and len(m.group(0)) >= 3 and m.group(0) not in bash_tokens
             and not any(m.group(0) in t for t in bash_tokens)]
    if not spans:
        return s
    for a, b in sorted(rng.sample(spans, min(n, len(spans))), reverse=True):
        w = list(s[a:b])
        i = rng.randrange(len(w))
        op = rng.choice(["swap", "delete", "double", "neighbor"])
        if op == "swap" and i < len(w) - 1:
            w[i], w[i + 1] = w[i + 1], w[i]
        elif op == "delete" and len(w) > 3:
            del w[i]
        elif op == "neighbor" and w[i].lower() in NEIGHBORS:
            w[i] = rng.choice(NEIGHBORS[w[i].lower()])
        else:
            w.insert(i, w[i])
        s = s[:a] + "".join(w) + s[b:]
    return s

def make_noisy(s, bash, rng):
    ops = [o for o, p in (("mix_en", 0.5), ("no_accent", 0.5), ("typo", 0.6)) if rng.random() < p]
    if not ops:
        ops = [rng.choice(["mix_en", "no_accent", "typo"])]
    out = s
    if "mix_en" in ops:
        out = mix_terms(out, rng)
    if "no_accent" in ops:
        out = strip_accents(out) if rng.random() < 0.7 else partial_no_accent(out, rng)
    if "typo" in ops:
        out = add_typos(out, rng, bash, n=rng.choice([1, 1, 2]))
    if out == s:
        out = add_typos(out, rng, bash, n=1)
        ops.append("typo_forced")
    if out == s:
        out = strip_accents(out)
        ops.append("no_accent_forced")
    return out, ops

def noisy_for(row_id, text, bash):
    return make_noisy(text, bash, random.Random(f"{SEED}-noise-{row_id}"))

def selected_for_noise(row_id):
    return random.Random(f"{SEED}-select-{row_id}").random() < NOISE_RATIO

s = "xoá tất cả file .log trong thư mục /var/log có kích thước lớn hơn 10MB"
for k in range(4):
    print(make_noisy(s, "find /var/log -name '*.log' -size +10M -delete", random.Random(k)))

('xoá tất cả file .log trong thư mục /var/log có kích thước lớớn hơn 10MB', ['typo'])
('remove tất cả file .log trong thư mục /var/log có size lớn hơn 10MB', ['mix_en'])
('xoá tất cả file .log trong thư mmục /var/log có kích thước lớn hơn 10MB', ['typo'])
('delete tất cả file .log trong thư mụcc /var/log có kích thước lớn hơn 10MB', ['mix_en', 'typo'])


### 1.4 Lọc và tạo tập cuối  (1–3 phút)
- Train: bỏ câu dịch không qua kiểm tra (rỗng, lẫn chữ lạ, không phải tiếng Việt, mất token cần giữ...). Lệnh đã qua bashlex ở 1.1 (kiểm tra lại cho chắc).
- Val: tách **theo nhóm lệnh** từ train đã lọc.
- Test: giữ mọi câu có bản dịch không rỗng và không lẫn chữ lạ; câu không qua kiểm tra token được **gắn cờ** `vi_check` (nên sửa tay qua file rà soát ở 1.2b) thay vì loại, để test giữ gần đủ 300 câu.

In [12]:
with timed("1.4_filter", LOGS):
    fstats = collections.Counter()
    train_rows = []
    for r in en_train:
        v = vi_by_id[r["id"]]
        if not v["ok"]:
            fstats[f"train_drop_{v['reason']}"] += 1
            continue
        if not parse_ok(r["bash"]):
            fstats["train_drop_bashlex"] += 1
            continue
        clean = v["nl_vi"]
        noisy, ops = noisy_for(r["id"], clean, r["bash"]) if selected_for_noise(r["id"]) else (None, [])
        train_rows.append({"id": r["id"], "nl": r["nl"], "bash": r["bash"],
                           "nl_vi_clean": clean, "nl_vi_noisy": noisy, "noise_ops": ops})

    test_rows = []
    for r in en_test:
        v = vi_by_id[r["id"]]
        if not v["nl_vi"] or v["reason"] in ("empty", "cjk"):
            fstats[f"test_drop_{v['reason']}"] += 1
            continue
        noisy, ops = noisy_for(r["id"], v["nl_vi"], r["bash"])
        test_rows.append({**r, "nl_vi_clean": v["nl_vi"], "nl_vi_noisy": noisy, "noise_ops": ops,
                          "vi_check": v["reason"]})

    groups = sorted({normalize_cmd(r["bash"]) for r in train_rows})
    random.Random(SEED).shuffle(groups)
    n_val_groups = max(1, int(round(VAL_FRAC * len(groups)))) if not SMOKE else max(1, len(groups) // 10)
    val_groups = set(groups[:n_val_groups])
    final_val = [r for r in train_rows if normalize_cmd(r["bash"]) in val_groups]
    final_train = [r for r in train_rows if normalize_cmd(r["bash"]) not in val_groups]
    final_test = test_rows

    def to_sft(rows):
        out = []
        for r in rows:
            if NOISY_MODE == "replace" and r["nl_vi_noisy"]:
                out.append({"id": r["id"], "nl_vi": r["nl_vi_noisy"], "bash": r["bash"], "variant": "noisy"})
                continue
            out.append({"id": r["id"], "nl_vi": r["nl_vi_clean"], "bash": r["bash"], "variant": "clean"})
            if r["nl_vi_noisy"]:
                out.append({"id": r["id"] + "-n", "nl_vi": r["nl_vi_noisy"], "bash": r["bash"], "variant": "noisy"})
        random.Random(SEED).shuffle(out)
        return out

    sft_train, sft_val = to_sft(final_train), to_sft(final_val)
    for name, rows in [("final_train", final_train), ("final_val", final_val), ("final_test", final_test),
                       ("sft_train", sft_train), ("sft_val", sft_val)]:
        write_jsonl(DATA / f"{name}.jsonl", rows)

    fstats.update({"final_train": len(final_train), "final_val": len(final_val), "final_test": len(final_test),
                   "train_with_noisy": sum(r["nl_vi_noisy"] is not None for r in final_train),
                   "sft_train_examples": len(sft_train), "sft_val_examples": len(sft_val),
                   "test_flagged_vi_check": sum(r["vi_check"] not in ("ok", "human_fixed") for r in final_test)})
    write_json(RES / "filter_stats.json", dict(fstats))

print(json.dumps(dict(fstats), ensure_ascii=False, indent=2))
print(pd.DataFrame(final_test[:5])[["nl", "nl_vi_clean", "nl_vi_noisy", "bash"]].to_string())

⏱ 1.4_filter: 0.0 phút
{
  "train_drop_missing_token": 2,
  "train_drop_length_ratio": 1,
  "train_drop_cjk": 1,
  "final_train": 42,
  "final_val": 4,
  "final_test": 50,
  "train_with_noisy": 6,
  "sft_train_examples": 48,
  "sft_val_examples": 5,
  "test_flagged_vi_check": 1
}
                                                                                                                                                                     nl                                                                                                                                                         nl_vi_clean                                                                                                                                                          nl_vi_noisy                                                                                                                                                                              bash
0                                           

### 1.5 SFT LoRA `Qwen2.5-Coder-1.5B-Instruct`  (A100: ~30–50 phút; T4 16GB: ~70–110 phút cho 2 epoch; SMOKE: 20 bước, ~2–8 phút)
- trl `SFTTrainer`, dữ liệu dạng hội thoại prompt/completion → **chỉ tính loss trên lệnh bash** (không học lại câu hỏi).
- LoRA r=16, alpha=32 trên mọi lớp linear; **fp16** (T4 không hỗ trợ bf16 phần cứng); batch hiệu dụng 64 (batch nhỏ hơn, gradient accumulation cao hơn trên T4); lr 2e-4 cosine.
- Checkpoint mỗi 200 bước vào `checkpoints/<run>/sft_qwen25coder15b_lora/` (giữ 3 bản gần nhất). Chạy lại ô → tự resume từ checkpoint mới nhất; nếu đã có `final/` thì bỏ qua.

In [13]:
%%writefile /kaggle/working/vishell_scripts/train_sft.py
import sys, json, math
from pathlib import Path
from vs_utils import read_jsonl, write_json, latest_checkpoint, hf_dtype_kw

def main():
    cfg = json.load(open(sys.argv[1]))
    out = Path(cfg["out_dir"])
    final = out / "final"
    if (final / "adapter_config.json").exists():
        print(f"[sft] Đã có adapter cuối ở {final} — bỏ qua.")
        return
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig

    sp = cfg["system_prompt"]
    def to_ex(r):
        return {"prompt": [{"role": "system", "content": sp}, {"role": "user", "content": r["nl_vi"]}],
                "completion": [{"role": "assistant", "content": r["bash"]}] }
    train = Dataset.from_list([to_ex(r) for r in read_jsonl(cfg["train"])])
    val = Dataset.from_list([to_ex(r) for r in read_jsonl(cfg["val"])])

    tok = AutoTokenizer.from_pretrained(cfg["base_model"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(cfg["base_model"], attn_implementation="sdpa",
                                                 **hf_dtype_kw(torch.float16))

    steps_per_epoch = math.ceil(len(train) / (cfg["per_device_bs"] * cfg["grad_accum"]))
    total = cfg["max_steps"] if cfg["max_steps"] > 0 else steps_per_epoch * cfg["epochs"]
    args = SFTConfig(
        output_dir=str(out), num_train_epochs=cfg["epochs"], max_steps=cfg["max_steps"],
        per_device_train_batch_size=cfg["per_device_bs"], per_device_eval_batch_size=cfg["per_device_bs"],
        gradient_accumulation_steps=cfg["grad_accum"], learning_rate=cfg["lr"], lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.03 * total)), logging_steps=10,
        save_strategy="steps", save_steps=cfg["save_steps"], save_total_limit=3,
        eval_strategy="steps", eval_steps=cfg["eval_steps"],
        fp16=True, max_length=cfg["max_len"], completion_only_loss=True,
        report_to="none", seed=cfg["seed"], dataloader_num_workers=2)
    peft_cfg = LoraConfig(r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"], lora_dropout=cfg["lora_dropout"],
                          target_modules="all-linear", task_type="CAUSAL_LM")
    trainer = SFTTrainer(model=model, args=args, train_dataset=train, eval_dataset=val,
                         processing_class=tok, peft_config=peft_cfg)
    trainer.model.print_trainable_parameters()

    ckpt = latest_checkpoint(out)
    print(f"[sft] {len(train)} mẫu train, {len(val)} val, ~{total} bước. Resume từ: {ckpt}", flush=True)
    trainer.train(resume_from_checkpoint=ckpt)
    trainer.save_model(str(final))
    tok.save_pretrained(str(final))
    write_json(out / "log_history.json", trainer.state.log_history)
    print(f"[sft] Xong. Adapter: {final}", flush=True)

if __name__ == "__main__":
    main()

Writing /kaggle/working/vishell_scripts/train_sft.py


In [14]:
SFT_DIR = CKPT / "sft_qwen25coder15b_lora"
sft_cfg = dict(base_model=SFT_BASE_MODEL, train=str(DATA / "sft_train.jsonl"), val=str(DATA / "sft_val.jsonl"),
               out_dir=str(SFT_DIR), system_prompt=SYSTEM_PROMPT,
               lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
               epochs=EPOCHS, lr=LR, max_len=MAX_LEN, seed=SEED,
               per_device_bs=8 if SMOKE else TRAIN_BS, grad_accum=1 if SMOKE else GRAD_ACC,
               max_steps=20 if SMOKE else -1,
               save_steps=10 if SMOKE else 200, eval_steps=10 if SMOKE else 200)
run_py(SCRIPTS / "train_sft.py", sft_cfg, LOGS, "sft")

hist = read_json(SFT_DIR / "log_history.json", [])
ev = [h for h in hist if "eval_loss" in h]
if ev:
    print("eval_loss:", [(h["step"], round(h["eval_loss"], 4)) for h in ev])

W0926 07:16:54.947000 681 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0926 07:16:54.988000 681 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 213.96it/s]

Tokenizing train dataset: 100%|██████████| 48/48 [00:00<00:00, 717.75 examples/s]

Building labels for train dataset: 100%|██████████| 48/48 [00:00<00:00, 4191.95 examples/s]

Truncating train dataset: 100%|██████████| 48/48 [00:00<00:00, 3903.87 examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]
Dropping fully mask

### 1.6 Đánh giá (không thực thi lệnh)  (A100: ~4–8 phút; T4 16GB: ~10–18 phút; SMOKE: ~2–8 phút)
Sinh dự đoán bằng vLLM (greedy), cùng một system prompt cho model gốc và model sau SFT (LoRA nạp qua `LoRARequest`). Ba biến thể đầu vào: `clean`, `noisy`, và `en` (câu tiếng Anh gốc — làm mốc tham chiếu).

**Chỉ số**
- `parse_rate`: tỉ lệ lệnh sinh ra parse được bằng bashlex.
- `exact_match`: trùng khớp sau chuẩn hoá khoảng trắng với **`bash` hoặc `bash2`** (test có 2 đáp án đúng).
- `utility_acc`: tiện ích chính trùng với tiện ích chính của một trong hai đáp án. Tiện ích chính = từ lệnh đầu tiên của simple command đầu tiên, bỏ qua `sudo/env/time/nohup/...` và phép gán biến (vd `sudo find . | xargs rm` → `find`).

EM đánh giá thấp mô hình (lệnh tương đương nhưng viết khác bị tính sai) — bài 2502.06858 dùng đánh giá bằng thực thi + LLM cho mục này; ở đây chỉ đánh giá tĩnh theo yêu cầu.

In [15]:
%%writefile /kaggle/working/vishell_scripts/eval_gen.py
import sys, json, os
from vs_utils import read_jsonl, write_jsonl, extract_command

def main():
    cfg = json.load(open(sys.argv[1]))
    rows = read_jsonl(cfg["test"])
    out_path = cfg["out"]
    preds = read_jsonl(out_path) if os.path.exists(out_path) else []
    done = {(p["system"], p["variant"], p["id"]) for p in preds}
    blocks = [(s, v, f) for s in cfg["systems"] for v, f in cfg["variants"].items()]
    todo_blocks = [(s, v, f) for s, v, f in blocks if any((s, v, r["id"]) not in done for r in rows)]
    print(f"[eval] {len(rows)} câu test, còn {len(todo_blocks)}/{len(blocks)} khối cần sinh", flush=True)
    if not todo_blocks:
        return
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    use_lora = "sft" in cfg["systems"]
    llm = LLM(model=cfg["base_model"], dtype="float16", max_model_len=1024,
              gpu_memory_utilization=cfg["gpu_mem"], seed=cfg["seed"],
              enable_lora=use_lora, max_lora_rank=cfg["lora_rank"] if use_lora else 16)
    sp = SamplingParams(temperature=0.0, max_tokens=cfg["max_tokens"])
    lora = LoRARequest("sft", 1, cfg["lora_path"]) if use_lora else None
    for system, variant, field in todo_blocks:
        todo = [r for r in rows if (system, variant, r["id"]) not in done]
        convs = [[{"role": "system", "content": cfg["system_prompt"]}, {"role": "user", "content": r[field]}]
                 for r in todo]
        outs = llm.chat(convs, sp, use_tqdm=True, lora_request=lora if system == "sft" else None)
        for r, o in zip(todo, outs):
            raw = o.outputs[0].text
            preds.append({"id": r["id"], "system": system, "variant": variant, "input": r[field],
                          "raw": raw, "pred": extract_command(raw)})
        write_jsonl(out_path, preds)       # lưu sau mỗi khối → resume được
        print(f"[eval] xong {system}/{variant}", flush=True)

if __name__ == "__main__":
    main()

Writing /kaggle/working/vishell_scripts/eval_gen.py


In [16]:
PRED_PATH = RES / "predictions.jsonl"
ev_cfg = dict(base_model=SFT_BASE_MODEL, lora_path=str(SFT_DIR / "final"), lora_rank=LORA_R,
              test=str(DATA / "final_test.jsonl"), out=str(PRED_PATH), systems=["base", "sft"],
              variants={"clean": "nl_vi_clean", "noisy": "nl_vi_noisy", "en": "nl"},
              system_prompt=SYSTEM_PROMPT, max_tokens=128, gpu_mem=0.6, seed=SEED)
run_py(SCRIPTS / "eval_gen.py", ev_cfg, LOGS, "eval_gen")

[eval] 50 câu test, còn 6/6 khối cần sinh
W0926 07:18:16.528000 805 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0926 07:18:16.569000 805 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
INFO 09-26 07:18:22 [api_utils.py:286] non-default args: {'dtype': 'float16', 'seed': 42, 'max_model_len': 1024, 'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'enable_lora': True, 'model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct'}
INFO 09-26 07:18:39 [model.py:692] Resolved architecture: Qwen2ForCausalLM
WARNING 09-26 07:18:39 [model.py:2364] Casting torch.bfloat16 to torch.float16.
INFO 09-26 0

226.28859901428223

In [17]:
with timed("1.6_metrics", LOGS):
    final_test = read_jsonl(DATA / "final_test.jsonl")
    test_by_id = {r["id"]: r for r in final_test}
    recs = []
    for p in read_jsonl(PRED_PATH):
        t = test_by_id[p["id"]]
        golds = [g for g in {t["bash"], t.get("bash2") or t["bash"]} if g]
        gold_norm = {normalize_cmd(g) for g in golds}
        gold_util = {main_utility(g) for g in golds} - {None}
        pu = main_utility(p["pred"])
        recs.append({"system": p["system"], "variant": p["variant"], "difficulty": t.get("difficulty", -1),
                     "parse": parse_ok(p["pred"]), "em": normalize_cmd(p["pred"]) in gold_norm,
                     "util": pu is not None and pu in gold_util})
    df = pd.DataFrame(recs)
    summary = (df.groupby(["system", "variant"])
                 .agg(n=("em", "size"), parse_rate=("parse", "mean"),
                      exact_match=("em", "mean"), utility_acc=("util", "mean"))
                 .reset_index())
    by_diff = (df[df["difficulty"] >= 0]
                 .groupby(["system", "variant", "difficulty"])
                 .agg(n=("em", "size"), exact_match=("em", "mean"), utility_acc=("util", "mean"))
                 .reset_index())
    gold_parse = float(pd.Series([parse_ok(r["bash"]) for r in final_test]).mean())

    order = {"clean": 0, "noisy": 1, "en": 2}
    summary = summary.sort_values(["variant", "system"], key=lambda s: s.map(order) if s.name == "variant" else s)
    show = summary.copy()
    for c in ("parse_rate", "exact_match", "utility_acc"):
        show[c] = (show[c] * 100).round(1)
    print(f"Test: {len(final_test)} câu | parse_rate của lệnh gold (mốc bashlex): {gold_parse * 100:.1f}%\n")
    print(show.to_string(index=False))
    if len(by_diff):
        piv = by_diff.pivot_table(index=["system", "variant"], columns="difficulty", values="exact_match")
        print("\nExact match theo độ khó (0=dễ, 1=vừa, 2=khó):")
        print((piv * 100).round(1).to_string())

    write_json(RES / "metrics.json", {
        "run": RUN, "smoke": SMOKE, "test_source": read_json(RES / "data_stats.json", {}).get("test_source"),
        "n_test": len(final_test), "gold_bashlex_parse_rate": gold_parse,
        "summary": summary.to_dict(orient="records"), "by_difficulty": by_diff.to_dict(orient="records"),
        "config": {"translator": TRANSLATOR_MODEL, "sft_base": SFT_BASE_MODEL, "lora_r": LORA_R,
                   "lora_alpha": LORA_ALPHA, "epochs": EPOCHS, "lr": LR, "noise_ratio": NOISE_RATIO,
                   "noisy_mode": NOISY_MODE, "seed": SEED},
        "data_stats": read_json(RES / "data_stats.json", {}), "filter_stats": read_json(RES / "filter_stats.json", {}),
    })
    print("\nĐã lưu", RES / "metrics.json", "và", PRED_PATH)

# vài ví dụ SFT sai trên bản nhiễu
_preds = {(p["system"], p["variant"], p["id"]): p for p in read_jsonl(PRED_PATH)}
_bad = [k for k in _preds if k[0] == "sft" and k[1] == "noisy"
        and normalize_cmd(_preds[k]["pred"]) not in {normalize_cmd(test_by_id[k[2]]["bash"]), normalize_cmd(test_by_id[k[2]]["bash2"])}]
for k in _bad[:5]:
    p, t = _preds[k], test_by_id[k[2]]
    print(f"\nIN  : {p['input']}\nPRED: {p['pred']}\nGOLD: {t['bash']}   |   {t['bash2']}")

Test: 50 câu | parse_rate của lệnh gold (mốc bashlex): 98.0%

system variant  n  parse_rate  exact_match  utility_acc
  base   clean 50        98.0         32.0         62.0
   sft   clean 50        98.0         24.0         66.0
  base   noisy 50       100.0         20.0         56.0
   sft   noisy 50       100.0         14.0         56.0
  base      en 50        98.0         40.0         74.0
   sft      en 50        98.0         38.0         80.0

Exact match theo độ khó (0=dễ, 1=vừa, 2=khó):
difficulty         0     1     2
system variant                  
base   clean    55.0  21.4  12.5
       en       75.0  21.4  12.5
       noisy    35.0   7.1  12.5
sft    clean    45.0  14.3   6.2
       en       75.0  21.4   6.2
       noisy    20.0  14.3   6.2

Đã lưu /kaggle/working/vishell/results/smoke/metrics.json và /kaggle/working/vishell/results/smoke/predictions.jsonl
⏱ 1.6_metrics: 0.0 phút

IN  : hien tthi file bash profile trong thu muc nhà
PRED: source ~/.bash_profile
GOLD: cat ~

### 1.7 Demo  (~1 phút)
Nạp model gốc + adapter LoRA bằng transformers/peft trong kernel chính. **Chỉ in lệnh ra màn hình, không thực thi.** Đặt `DEMO_INTERACTIVE = True` ở ô cấu hình để nhập câu tự do (Enter trống để thoát). Chạy ô cuối để giải phóng GPU.

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

_tok = AutoTokenizer.from_pretrained(SFT_BASE_MODEL)
_base = AutoModelForCausalLM.from_pretrained(SFT_BASE_MODEL, device_map="cuda", **hf_dtype_kw(torch.float16))
demo_model = PeftModel.from_pretrained(_base, str(SFT_DIR / "final")).eval()

def sinh_lenh(cau, max_new_tokens=128):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": cau}]
    text = _tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = _tok(text, return_tensors="pt").to(demo_model.device)
    with torch.no_grad():
        out = demo_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=_tok.pad_token_id or _tok.eos_token_id)
    return extract_command(_tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))

def hien_thi(cau):
    cmd = sinh_lenh(cau)
    print(f"🗣  {cau}\n💻 {cmd}\n   (parse={parse_ok(cmd)}, tiện ích={main_utility(cmd)}) — chỉ hiển thị, KHÔNG thực thi\n")

for cau in ["liệt kê tất cả file .log trong /var/log lớn hơn 10MB",
            "xoa het file .tmp trong thu muc hien tai",
            "đếm số dòng của file data.csv",
            "tìm process đang dùng port 8080",
            "nén folder project thành project.tar.gz"]:
    hien_thi(cau)

if DEMO_INTERACTIVE:
    while True:
        cau = input("Nhập yêu cầu (Enter trống để thoát): ").strip()
        if not cau:
            break
        hien_thi(cau)

W0926 07:22:02.139000 23 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0926 07:22:02.182000 23 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

🗣  liệt kê tất cả file .log trong /var/log lớn hơn 10MB
💻 find /var/log -name "*.log" -size +10M
   (parse=True, tiện ích=find) — chỉ hiển thị, KHÔNG thực thi

🗣  xoa het file .tmp trong thu muc hien tai
💻 find / -name "*.tmp" -type f -exec rm {} \;
   (parse=True, tiện ích=find) — chỉ hiển thị, KHÔNG thực thi

🗣  đếm số dòng của file data.csv
💻 wc -l data.csv
   (parse=True, tiện ích=wc) — chỉ hiển thị, KHÔNG thực thi

🗣  tìm process đang dùng port 8080
💻 lsof -i :8080
   (parse=True, tiện ích=lsof) — chỉ hiển thị, KHÔNG thực thi

🗣  nén folder project thành project.tar.gz
💻 tar -czvf project.tar.gz project
   (parse=True, tiện ích=tar) — chỉ hiển thị, KHÔNG thực thi



In [19]:
del demo_model, _base
free_gpu()

GPU đang dùng: 3.0 GB
